In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import ast
from datetime import timedelta
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={os.getenv('DB_SERVER')};"
    "DATABASE=master;"
    f"UID={os.getenv('DB_USER')};"
    f"PWD={os.getenv('DB_PASSWORD')};"
)

# Crear el motor de conexión
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [75]:
query_com = """
SELECT DISTINCT Pedido,
    c.Cliente,
    c.Nombres + ' ' + c.Apellidos AS Nombre,
    c.Email AS Email,
    c.Celular AS Celular,
    v.Fecha AS Fecha_Venta,
    v.Canal,
    SUM(v.Venta_Neta) AS Venta
FROM Ventas_Comerssia.dbo.Contacto_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.fecha >= '2026-02-27'
    AND v.Fecha <= '2026-02-28'
    AND Canal IN ('Shopify')
    AND v.Venta_Neta > 0
GROUP BY  Pedido,
    c.Cliente,
    c.Documento,
    c.Nombres + ' ' + c.Apellidos,
    c.Email,
    c.Celular,
    v.Fecha,
    v.Canal
"""

# Ejecutar y cargar en DataFrame
df_ventas = pd.read_sql(query_com, engine)

df_ventas.head()

,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta
0,96684,C283486,ARTURO MURGUEYTIO,arturomurgueytio@icloud.com,3147319587,2026-02-27,Shopify,215116.8
1,96689,C52388222,ELBA RODRIGUEZ,elbarsan@gmail.com,3153046000,2026-02-27,Shopify,265534.8
2,96691,C36309117,LEIDY ESCOBAR,leidyjes@gmail.com,3114687382,2026-02-27,Shopify,108398.7
3,96692,C42121364,FRANCIA ELENA RIVERA,frivera48@hotmail.com,3104698312,2026-02-27,Shopify,226881.0
4,96694,C1090482626,RUBEN TEULING,ruben@teulinglira.com,3058380611,2026-02-27,Shopify,325196.1


In [76]:
# #consulta segmentada 
# query_data = """
# SELECT DISTINCT c.Cliente,
#     c.Documento,
#     c.Nombres + ' ' + c.Apellidos AS Nombre,
#     c.Email AS Email,
#     c.Celular AS Celular,
#     v.Fecha AS Fecha_Venta,
#     v.Canal,
#     SUM(v.Venta_Neta) AS Venta
# FROM Ventas_Comerssia.dbo.View_Contacto_Clientes c
# INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
# WHERE v.fecha >= '2025-10-02'
#     AND v.Fecha <= '2025-10-07'
#     AND Canal IN ('Shopify','CHATCENTER WEB')
#     AND v.Venta_Neta > 0
#     AND Codigo_Descuento = 'RECOMPRA-ACEITE'
# GROUP BY  c.Cliente,
#          c.Documento,
#        c.Nombres + ' ' + c.Apellidos,
#        c.Email,
#        c.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)

# df_ventas.head()

In [77]:
# # CONSULTAS CUMPLEAÑOS

# query_data = """
# SELECT DISTINCT v.Cliente, 
# 	cc.Nombres + ' ' +  cc.Apellidos AS Nombre, 
# 	cc.Email, cc.Celular, 
# 	v.Fecha AS Fecha_Venta 
# 	,SUM(Venta_Neta) AS Venta, 
# 	Canal
# FROM Ventas_Comerssia.dbo.Ventas_Unificadas v
# INNER JOIN Ventas_Comerssia.dbo.View_Contacto_Clientes cc ON cc.Cliente = v.Cliente
# WHERE Codigo_Descuento LIKE '%Cump%'
# 	AND Fecha BETWEEN '2025-07-04' AND '2025-07-31'
# 	AND NOT EXISTS (SELECT 1 FROM COMERSSIA.PROVENZAL.dbo.EmpleadosProvenzal e WHERE e.Codigo = v.cliente ) 
# GROUP BY  v.Cliente,
#        cc.Nombres + ' ' +  cc.Apellidos,
#        cc.Email,
#        cc.Celular,
#        v.Fecha,
#        v.Canal
# """

# # Ejecutar y cargar en DataFrame
# df_ventas = pd.read_sql(query_data, engine)
# df_ventas.head()

In [78]:
# Leer archivo
df_revie = pd.read_excel("Atribucion.xlsx")

# Seleccionar columnas relevantes
df_campania = df_revie[['Email', 'First Name', 'Last Name', 'Phone', 'Company']].copy()

# 4. Normalizar correos
df_campania['Email'] = df_campania['Email'].str.lower().str.strip()


In [79]:
# Normalizar df_ventas
df_ventas['Email'] = df_ventas['Email'].str.lower().str.strip()
df_ventas['Fecha_Venta'] = pd.to_datetime(df_ventas['Fecha_Venta']).dt.date

# Merge por email
df_atribucion = pd.merge(df_ventas, df_campania, on='Email', how='inner')

# Sumar la venta total atribuida
total_venta = df_atribucion['Venta'].sum()

filas = len(df_atribucion) # Ver cuántas filas hay realmente

print(f"Total de Ventas: {filas}")
print(f"Total de venta atribuida: {total_venta:,.2f}"  )
# Resultado final
df_atribucion.head()

Total de Ventas: 3
Total de venta atribuida: 1,721,774.70


,Pedido,Cliente,Nombre,Email,Celular,Fecha_Venta,Canal,Venta,First Name,Last Name,Phone,Company
0,96706,C1010224919,LAURA TATIANA MUÑOZ TORRES,ltatiana05@hotmail.com,3105805972,2026-02-27,Shopify,635266.8,LAURA TATIANA,MUÑOZ TORRES,3105805972,1010224919
1,96715,C30394396,ANDREA OCAMPO SALGADO,aocamposalgado@gmail.com,3205237738,2026-02-27,Shopify,443678.4,ANDREA,OCAMPO SALGADO,3205237738,30394396
2,96719,C22585866,KAREN TORRENEGRA,ktorrenegra@hotmail.com,3102487940,2026-02-27,Shopify,642829.5,KAREN,TORRENEGRA,3102487940,22585866


In [80]:
df_atribucion.to_excel('Ventas.xlsx', index=False)